# ARCS v0.6.3: a fresh circular benchmark

This notebook computes a new 256-panel circular BEM solve at 100 frequencies, 10 Hz–10 THz. It reads no saved spectra and uses no variables from the longer notebook. Restart the kernel after installing, then Run All. The reference uses independent cylindrical Maxwell matching.


**Saved output provenance:** these completed calculations were executed with ARCS v0.6.1. The code below runs with v0.6.3. Notebook 04 contains a fresh v0.6.3 calculation with axial pipe length.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys, platform
import numpy as np
import scipy
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / "src/arcs").exists():
    ROOT = ROOT.parent
if (ROOT / "src/arcs").exists():
    sys.path.insert(0, str(ROOT / "src"))
import arcs
from arcs import load_geometry, Metal, Wall
from arcs.reference import circular_exact
from arcs.plotting import plot_longitudinal
from arcs.special import decaying_k
print("Python", platform.python_version(), "NumPy", np.__version__, "SciPy", scipy.__version__)
print("ARCS", arcs.__version__, arcs.__file__)
assert arcs.__version__ == "0.6.6", "Install ARCS v0.6.6, restart the kernel, then Run All."
print("Stable K0(680−680j):", decaying_k(0, 680-680j))


In [ ]:
circle_definition = {
    "units": "mm", "aperture": {"type": "circle", "radius": 30},
    "mesh": {"minimum_panels": 256, "max_panel_length": 1., "curve_tolerance": .01},
    "regions": [{"name": "beam", "material": "vacuum"}, {"name": "wall", "material": "Cu"}],
    "materials": {"Cu": {"conductivity": 5.8e7}}
}
circle = load_geometry(circle_definition)
circular_solver = circle.make_solver("maxwell", pipe_length_m=None)
f_circle = np.geomspace(10., 1e13, 100)
print("Actual panels:", len(circular_solver.backend.panels.panel_start))
z_circle = circular_solver.longitudinal(f_circle)
z_circle_reference = circular_exact(f_circle, .03, Wall(Metal(5.8e7)))
print("Maximum complex error [%]:", 100*np.max(abs(z_circle/z_circle_reference - 1)))
print("Maximum Re-component error [%]:", 100*np.max(abs(z_circle.real/z_circle_reference.real - 1)))
assert np.max(abs(z_circle/z_circle_reference - 1)) < .005
assert np.max(abs(z_circle.real/z_circle_reference.real - 1)) < .005


In [ ]:
figure = plot_longitudinal(f_circle, z_circle, reference=z_circle_reference,
                          title="ARCS v0.6.0 · fresh 30 mm copper circle")
# For linear impedance axes:
# for ax in figure.axes:
#     ax.set_yscale("linear")
display(figure)
plt.close(figure)
figure, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
axes[0].semilogx(f_circle, 100*(z_circle.real/z_circle_reference.real-1))
axes[0].set(ylabel="Re-component relative error [%]", xlabel="Frequency [Hz]")
axes[1].semilogx(f_circle, z_circle.imag-z_circle_reference.imag)
axes[1].set(ylabel="Signed Im difference [Ω/m]", xlabel="Frequency [Hz]")
for ax in axes:
    ax.grid(alpha=.2)
display(figure)
plt.close(figure)
